# 🎵 MelodyChief — Musician Recommendation Engine

**Hybrid ML System**: Content-Based Filtering + Collaborative Filtering

| Algorithm | Description |
|---|---|
| **Content-Based** | Filters musicians by instrument, genre, and skill level |
| **Collaborative** | Cosine similarity on feature vectors to find similar musicians |
| **Hybrid** | Union of both, re-ranked by combined score |

**Dataset**: `data/musicians.json` — 50 curated musician profiles (mirrors Hackathon dataset schema)

---


In [ ]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries loaded")


In [ ]:
# ── Load dataset ──────────────────────────────────────────────────
# Try local data/musicians.json first, fallback to hackathon_dataset.xlsx (Colab)

DATASET_JSON = os.path.join(os.path.dirname(os.getcwd()) if 'content' not in os.getcwd() else '/content',
                            'data', 'musicians.json')
DATASET_XLSX = '/content/hackathon_dataset.xlsx'

if os.path.exists(DATASET_JSON):
    with open(DATASET_JSON, 'r') as f:
        records = json.load(f)
    df = pd.DataFrame(records)
    print(f"✓ Loaded from musicians.json  ({len(df)} records)")
elif os.path.exists(DATASET_XLSX):
    df = pd.read_excel(DATASET_XLSX)
    print(f"✓ Loaded from hackathon_dataset.xlsx  ({len(df)} records)")
else:
    # Create inline sample for demonstration
    df = pd.DataFrame([
        {"id":1,"name":"Arjun Mehra","genre":"Jazz","skill_level":"Expert","location":"Mumbai",
         "instr1":"Piano","instr2":"Synthesizers","availability":"Weekends","profile_link":""},
        {"id":2,"name":"Priya Sharma","genre":"Classical","skill_level":"Advanced","location":"Delhi",
         "instr1":"Violin","instr2":"Viola","availability":"Evenings","profile_link":""},
        {"id":3,"name":"Rohan Das","genre":"Rock","skill_level":"Intermediate","location":"Bangalore",
         "instr1":"Guitar","instr2":"Bass Guitar","availability":"Full-time","profile_link":""},
        {"id":4,"name":"Sneha Patel","genre":"Pop","skill_level":"Advanced","location":"Pune",
         "instr1":"Vocals","instr2":"Piano","availability":"Weekends","profile_link":""},
        {"id":5,"name":"Vikram Singh","genre":"Fusion","skill_level":"Expert","location":"Chennai",
         "instr1":"Tabla","instr2":"Mridangam","availability":"Flexible","profile_link":""},
    ])
    print(f"⚠ Using built-in sample data  ({len(df)} records)")

# Ensure consistent dtypes
for col in ['instr1', 'instr2', 'genre', 'skill_level', 'location']:
    if col in df.columns:
        df[col] = df[col].fillna('').astype(str)

df.head()


,id,name,genre,location,skill_level,instr1,instr2,profile_link,followers,tracks,availability
0,1,Aarav Patel,Indie Folk,Mumbai,Advanced,Guitar,Vocals,https://soundcloud.com/aarav,2000,15,Yes
1,2,Neha Sharma,Classical,Delhi,Intermediate,Violin,Piano,https://soundcloud.com/neha,1500,10,No
2,3,Raj Singh,Hip-Hop,Bangalore,Beginner,Rapping,Beatboxing,https://soundcloud.com/raj,800,8,Yes
3,4,Priya Verma,Pop,Hyderabad,Advanced,Vocals,Guitar,https://soundcloud.com/priya,3000,20,Yes
4,5,Vikram Rao,Rock,Chennai,Intermediate,Electric Guitar,NaN,https://soundcloud.com/vikram,1200,12,No


In [ ]:
# ── Dataset overview ──────────────────────────────────────────────
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nSkill level distribution:")
print(df['skill_level'].value_counts().to_string())
print("\nTop genres:")
print(df['genre'].value_counts().head(8).to_string())


In [ ]:
# ── Feature engineering ───────────────────────────────────────────
SKILL_RANK = {'Beginner': 1, 'Intermediate': 2, 'Advanced': 3, 'Expert': 4}

def build_feature_matrix(dataframe):
    """Build numeric feature matrix for collaborative filtering."""
    # Encode genres as one-hot
    genre_dummies = pd.get_dummies(dataframe['genre'], prefix='genre')

    # Encode skill level as ordinal
    skill_encoded = dataframe['skill_level'].map(SKILL_RANK).fillna(2).astype(float) / 4.0

    # Encode experience (if exists)
    exp_col = dataframe.get('experience', pd.Series([5] * len(dataframe)))
    exp_norm = exp_col.fillna(5).astype(float).clip(0, 20) / 20.0

    # Combine
    feature_df = pd.concat([
        genre_dummies.reset_index(drop=True),
        skill_encoded.reset_index(drop=True).rename('skill'),
        exp_norm.reset_index(drop=True).rename('experience'),
    ], axis=1)
    return feature_df.fillna(0)

feature_matrix = build_feature_matrix(df)
print(f"✓ Feature matrix: {feature_matrix.shape}  ({feature_matrix.shape[1]} features per musician)")
feature_matrix.head(3)


In [ ]:
# ── Compute cosine similarity matrix ─────────────────────────────
similarity_matrix = cosine_similarity(feature_matrix)
similarity_df     = pd.DataFrame(
    similarity_matrix,
    index=df['name'].values,
    columns=df['name'].values
)
print(f"✓ Similarity matrix: {similarity_df.shape}")
print("\nTop 5 musicians similar to first entry:", df['name'].iloc[0])
sim_row = similarity_df.iloc[0].sort_values(ascending=False)
print(sim_row.head(6).to_string())


In [ ]:
# ── Content-Based Filtering ───────────────────────────────────────
def content_based_recommendations(dataframe, instrument=None, genre=None, skill=None, top_n=10):
    """
    Filter musicians by instrument (instr1/instr2), genre, and skill level.
    Returns top_n results sorted by skill rank descending.
    """
    mask = pd.Series([True] * len(dataframe), index=dataframe.index)

    if instrument:
        instr_mask = (
            dataframe['instr1'].str.contains(instrument, case=False, na=False) |
            dataframe['instr2'].str.contains(instrument, case=False, na=False)
        )
        mask &= instr_mask

    if genre:
        mask &= dataframe['genre'].str.contains(genre, case=False, na=False)

    if skill:
        mask &= dataframe['skill_level'].str.lower() == skill.lower()

    results = dataframe[mask].copy()
    results['_skill_rank'] = results['skill_level'].map(SKILL_RANK).fillna(2)
    return results.sort_values('_skill_rank', ascending=False).drop(columns=['_skill_rank']).head(top_n)

# Quick test
test = content_based_recommendations(df, instrument='Guitar', top_n=5)
print(f"Guitar players found: {len(test)}")
print(test[['name', 'instr1', 'instr2', 'genre', 'skill_level', 'location']].to_string(index=False))


In [ ]:
# ── Collaborative Filtering ───────────────────────────────────────
def collaborative_filtering_recommendations(dataframe, feat_matrix, sim_df,
                                             query_name=None, query_features=None, top_n=10):
    """
    Find musicians most similar to a given musician (by name) or a query feature vector.
    Uses pre-computed cosine similarity matrix.
    """
    if query_name and query_name in sim_df.index:
        sim_scores = sim_df[query_name].drop(index=query_name, errors='ignore')
    elif query_features is not None:
        # Compute on-the-fly similarity for a new query vector
        q_vec      = np.array(query_features).reshape(1, -1)
        raw_scores = cosine_similarity(q_vec, feat_matrix)[0]
        sim_scores = pd.Series(raw_scores, index=dataframe['name'].values)
    else:
        raise ValueError("Provide either query_name or query_features")

    top_names = sim_scores.sort_values(ascending=False).head(top_n).index.tolist()
    results   = dataframe[dataframe['name'].isin(top_names)].copy()
    results['_score'] = results['name'].map(sim_scores)
    return results.sort_values('_score', ascending=False).drop(columns=['_score'])

# Quick test — musicians similar to first entry
top_similar = collaborative_filtering_recommendations(df, feature_matrix, similarity_df,
                                                       query_name=df['name'].iloc[0], top_n=5)
print(f"Musicians similar to '{df['name'].iloc[0]}':")
print(top_similar[['name', 'instr1', 'genre', 'skill_level', 'location']].to_string(index=False))


In [ ]:
# ── Hybrid Recommendation Engine ──────────────────────────────────
def hybrid_recommendations(dataframe, feat_matrix, sim_df,
                            instrument=None, genre=None, skill=None, top_n=15):
    """
    Combines content-based and collaborative filtering.
    Content results get priority; collaborative fills remaining slots.
    """
    # 1. Content-based pass
    content_recs = content_based_recommendations(dataframe, instrument, genre, skill, top_n=top_n)
    content_names = set(content_recs['name'].values)

    # 2. Collaborative pass — use first content result as the "seed" user
    if not content_recs.empty:
        seed_name = content_recs.iloc[0]['name']
        collab_recs = collaborative_filtering_recommendations(
            dataframe, feat_matrix, sim_df, query_name=seed_name, top_n=top_n)
    else:
        collab_recs = pd.DataFrame(columns=dataframe.columns)

    # 3. Merge: content first, then collab (no duplicates)
    hybrid = pd.concat([content_recs, collab_recs], ignore_index=True)
    hybrid = hybrid.drop_duplicates(subset=['name']).reset_index(drop=True)

    # 4. Score column for transparency
    hybrid['algo'] = hybrid['name'].apply(
        lambda n: 'content' if n in content_names else 'collab')
    hybrid.loc[hybrid.index[:3], 'algo'] = hybrid.loc[hybrid.index[:3], 'algo'].apply(
        lambda a: 'hybrid' if a == 'collab' else a)

    return hybrid.head(top_n)

print("✓ Hybrid recommendation engine ready")


In [ ]:
# ── Similarity Heatmap (Top-20 musicians) ─────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

try:
    top20 = similarity_df.iloc[:20, :20]
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(top20, annot=False, cmap='magma', linewidths=0.3,
                xticklabels=top20.columns, yticklabels=top20.index, ax=ax)
    ax.set_title('Musician Similarity Matrix (Top 20)', fontsize=15, fontweight='bold')
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()
    plt.savefig('similarity_heatmap.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("✓ Similarity heatmap saved → similarity_heatmap.png")
except Exception as e:
    print(f"Visualization skipped: {e}")


In [ ]:
# ── Genre & Skill Distribution ─────────────────────────────────────
try:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Genre bar chart
    genre_counts = df['genre'].value_counts().head(10)
    axes[0].barh(genre_counts.index, genre_counts.values,
                 color=plt.cm.plasma(range(0, 256, 26)))
    axes[0].set_title('Top Genres', fontweight='bold')
    axes[0].set_xlabel('Count')

    # Skill pie chart
    skill_counts = df['skill_level'].value_counts()
    axes[1].pie(skill_counts.values, labels=skill_counts.index,
                autopct='%1.0f%%', startangle=140,
                colors=['#7c3aed', '#06b6d4', '#ec4899', '#10b981', '#f59e0b'])
    axes[1].set_title('Skill Distribution', fontweight='bold')

    plt.suptitle('MelodyChief — Dataset Overview', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('dataset_overview.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("✓ Dataset overview chart saved → dataset_overview.png")
except Exception as e:
    print(f"Chart skipped: {e}")


## Demo — Run Hybrid Recommendations

Try calling `hybrid_recommendations()` with any combination of filters.  
All three parameters are optional — omit any to widen the search.


In [ ]:
# ── Demo: Hybrid Recommendations ──────────────────────────────────
results = hybrid_recommendations(
    df, feature_matrix, similarity_df,
    instrument="Synthesizers",
    genre="Electronic",
    top_n=8
)

display_cols = ['name', 'genre', 'skill_level', 'location', 'instr1', 'instr2', 'algo']
available_cols = [c for c in display_cols if c in results.columns]
print("=== Top Recommendations ===")
print(results[available_cols].to_string(index=False))
print(f"\nTotal results: {len(results)}")


              name       genre   skill_level   location        instr1  \
0        Sara Khan  Electronic      Beginner  Bangalore  Synthesizers   
1      Shruti Nair  Electronic      Advanced     Mumbai  Synthesizers   
2     Puneet Singh         Pop  Intermediate     Mumbai        Vocals   
3     Sakshi Reddy         Pop  Intermediate  Bangalore        Vocals   
4      Kavya Singh         Pop  Intermediate       Pune        Vocals   
5  Siddharth Mehta         Pop      Advanced  Bangalore        Vocals   

          instr2 availability                            profile_link  
0         Vocals          Yes             https://soundcloud.com/sara  
1         Vocals          Yes           https://soundcloud.com/shruti  
2   Synthesizers           No           https://soundcloud.com/puneet  
3   Synthesizers           No           https://soundcloud.com/sakshi  
4   Synthesizers           No            https://soundcloud.com/kavya  
5   Synthesizers          Yes  https://soundcloud.com/si

In [ ]:
# ── Export recommendations + similarity matrix for the Web API ─────
import os, json

out_dir = os.path.join(os.path.dirname(os.path.abspath('.')), 'data') \
          if os.path.basename(os.getcwd()) != 'melody-chief' else 'data'
os.makedirs(out_dir, exist_ok=True)

# 1. Full recommendation output (all genres, no filter)
all_recs = hybrid_recommendations(df, feature_matrix, similarity_df, top_n=len(df))
export_cols = [c for c in ['id','name','genre','skill_level','location',
                            'instr1','instr2','availability','role','algo']
               if c in all_recs.columns]
out_path = os.path.join(out_dir, 'rec_output.json')
all_recs[export_cols].to_json(out_path, orient='records', indent=2)
print(f"✓ Recommendations exported → {out_path}  ({len(all_recs)} records)")

# 2. Similarity scores snapshot (top-10 for each musician)
sim_snapshot = {}
for name in similarity_df.index[:20]:
    top = similarity_df[name].sort_values(ascending=False)[1:6].to_dict()
    sim_snapshot[name] = {k: round(v, 4) for k, v in top.items()}

sim_path = os.path.join(out_dir, 'similarity_snapshot.json')
with open(sim_path, 'w') as f:
    json.dump(sim_snapshot, f, indent=2)
print(f"✓ Similarity snapshot exported → {sim_path}")
print("\nAll done! The Node.js server reads data/musicians.json directly.")
print("Run:  npm start   →   http://localhost:8080")
